# 08_sarima_forecasting.ipynb (Final)

**Objective:** Forecast Bitcoin prices using ARIMA and SARIMA, then save prediction comparisons for final reporting.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error
from datetime import timedelta
import warnings
import time
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/bitcoin_timeseries.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').set_index('timestamp')
df_hourly = df.resample('1h').mean().dropna()
train = df_hourly[:-24]
test = df_hourly[-24:]

## 🔹 ARIMA Forecast

In [ ]:
start = time.time()
model_arima = auto_arima(train, seasonal=False, stepwise=True, max_order=5, 
                         trace=True, error_action='ignore', suppress_warnings=True)
forecast_arima = model_arima.predict(n_periods=len(test))
print("ARIMA fit time: {:.2f}s".format(time.time() - start))

Performing stepwise search to minimize aic
 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=1509047.761, Time=20.80 sec
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=1516601.855, Time=1.25 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=1509524.961, Time=1.54 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=1509067.235, Time=3.49 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=1516603.620, Time=0.79 sec
 ARIMA(1,1,2)(0,0,0)[0] intercept   : AIC=1509067.436, Time=16.40 sec
 ARIMA(2,1,1)(0,0,0)[0] intercept   : AIC=1509055.330, Time=4.17 sec
 ARIMA(3,1,2)(0,0,0)[0] intercept   : AIC=1509047.863, Time=31.74 sec


## 🔹 SARIMA Forecast

In [ ]:
start = time.time()
model_sarima = auto_arima(train, seasonal=True, m=24, stepwise=True, max_order=5, 
                          trace=True, error_action='ignore', suppress_warnings=True)
forecast_sarima = model_sarima.predict(n_periods=len(test))
print("SARIMA fit time: {:.2f}s".format(time.time() - start))

## 📊 Forecast Comparison + Export

In [ ]:
last_timestamp = train.index[-1]
forecast_horizon = min(len(forecast_arima), len(forecast_sarima))
forecast_index = [last_timestamp + timedelta(hours=i) for i in range(1, forecast_horizon + 1)]
df_comparison = pd.DataFrame({
    'timestamp': forecast_index,
    'arima_pred': forecast_arima[:forecast_horizon],
    'sarima_pred': forecast_sarima[:forecast_horizon]
})
df_comparison.to_csv('../reports/arima_vs_sarima_forecast.csv', index=False)

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(forecast_index, forecast_arima[:forecast_horizon], label='ARIMA', marker='o')
plt.plot(forecast_index, forecast_sarima[:forecast_horizon], label='SARIMA', marker='x')
plt.title('ARIMA vs SARIMA Forecast Comparison')
plt.xlabel('Timestamp')
plt.ylabel('BTC Price')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
mae_arima = mean_absolute_error(test[:forecast_horizon], forecast_arima[:forecast_horizon])
rmse_arima = mean_squared_error(test[:forecast_horizon], forecast_arima[:forecast_horizon], squared=False)
mae_sarima = mean_absolute_error(test[:forecast_horizon], forecast_sarima[:forecast_horizon])
rmse_sarima = mean_squared_error(test[:forecast_horizon], forecast_sarima[:forecast_horizon], squared=False)

with open('../reports/arima_vs_sarima_metrics.txt', 'w') as f:
    f.write(f'ARIMA MAE: {mae_arima:.2f}, RMSE: {rmse_arima:.2f}\n')
    f.write(f'SARIMA MAE: {mae_sarima:.2f}, RMSE: {rmse_sarima:.2f}\n')